# Combining Structured and Unstructured Data
We will recreate an example use case from [LlamaIndex](https://docs.llamaindex.ai/en/stable/examples/query_engine/SQLJoinQueryEngine/).
In this example, we will create a model that will combine insights from structured data (SQL tables) and unstructured data (Wikipedia articles) to answer user queries. 

## Schema Creation and Loading Data
We first create a cities table which contains information about different cities regarding their geographic location, population and country in which they are located.


In [ ]:
CREATE TABLE langdb.cities (
	`id` UUID DEFAULT generateUUIDv4(),
	`city` String,
	`lat` Decimal64(3),
	`lng` Decimal64(3),
	`country` String,
	`population` UInt64
) engine = MergeTree
ORDER BY id;

In [ ]:
docker exec -i langdb clickhouse-client -q "INSERT INTO langdb.cities FORMAT CSVWithNames" < cities.csv

We will load a small [CSV](https://drive.google.com/file/d/1qTis3Ve7U-GWtR3J_1gNjhlsWGDKLC6I/view?usp=sharing) of 15 cities. 

In [13]:
SELECT city, lat, lng, country, population FROM langdb.cities;

,city,lat,lng,country,population
0,Karachi,24.860,67.010,Pakistan,15738000
1,Delhi,28.610,77.230,India,32226000
2,Beijing,39.906,116.397,China,18522000
3,Moscow,55.755,37.617,Russia,17332000
4,Dhaka,23.763,90.388,Bangladesh,18627000
5,Jakarta,-6.175,106.827,Indonesia,33756000
6,Manila,14.595,120.977,Philippines,24922000
7,Tokyo,35.689,139.692,Japan,37732000
8,Phnom Penh,11.569,104.921,Cambodia,2129371
9,Singapore,1.300,103.800,Singapore,5983000


## Loading the PDFs
We create a `pdfs` table to store the PDFs containing information about the cities, obtained from Wikipedia.
We extract semantically chunked data from the PDFs using built-in function, `load_pdf_text()`, and insert into the tables.

In [ ]:
CREATE TABLE pdf_links (
	`city` String,
	`link` String
) engine = MergeTree
order by city;

In [ ]:
CREATE TABLE pdfs (
  `id` UUID DEFAULT generateUUIDv4(),
  `content` String, 
  `metadata` String, 
  `city` String
) engine = MergeTree
order by (id, content);

[`pdf_links`](https://drive.google.com/file/d/1bK5UlLWmiwQ2-y1HNq4wSmjsgxqY_p3a/view?usp=sharing) has the links to the Wikipedia articles.

In [ ]:
docker exec -i langdb clickhouse-client -q "INSERT INTO pdf_links FORMAT CSVWithNames" < ./pdf_links.csv

In [ ]:
INSERT INTO pdfs(content, metadata, city) 
SELECT content, metadata, JSONExtractString(metadata,'city')
FROM load_pdf_text((SELECT link, city from pdf_links));

## Embeddings
LangDB offers a convenient method to generate embeddings using the built-in `embed()` function for development and testing purposes. Additionally, we can integrate with OpenAI or other providers to generate embeddings.

In [ ]:
CREATE TABLE pdf_embeddings (
  id UUID,
  embeddings `Array`(`Float32`),
) 
engine = MergeTree
order by id;

While we can use `embed()` to generate embeddings for each chunk and store them, we can also run the whole process in background using `SPAWN TASK` which essentially creates a cron job.

In [19]:
SPAWN TASK 
    BEGIN
INSERT INTO pdf_embeddings
select p.id, embed(content) from pdfs as p LEFT JOIN pdf_embeddings as pe on p.id = pe.id
where p.id != pe.id
order by p.id
limit 10
END EVERY 5 second;

Spawned Task: 5a94da1a-1156-476c-80fc-5647e6814735

## Endpoint Creation
First, we create two endpoints which utilize vector search to find relevant chunks. While the first the endpoint can be utilized to find information when the city is not known, the second endpoint can be used to query about a specific city.

In [ ]:
CREATE ENDPOINT cities_info_generic(query String "description of the information to look up about cities") AS
WITH tbl AS (
  SELECT CAST(embed($query) AS `Array`(`Float32`)) AS query
)
SELECT 
  p.id as id, 
  p.content as content, 
  cosineDistance(embeddings, query) AS cosineDistance,
  p.city as city
FROM 
  pdf_embeddings AS pe 
JOIN 
  pdfs AS p ON p.id = pe.id
CROSS JOIN 
  tbl 
ORDER BY 
  cosineDistance ASC
  LIMIT 5
WITH description = 'get information about cities';

In [ ]:
CREATE ENDPOINT cities_info_specific(
  city_name String "name of the city", 
  query String "a description of the information to look up about the city specified in the city_name parameter"
) AS
WITH tbl AS (
  SELECT CAST(embed($query) AS `Array`(`Float32`)) AS query
)
SELECT 
  p.id as id, 
  p.content as content, 
  cosineDistance(embeddings, query) AS cosineDistance,
  p.city as city
FROM 
  pdf_embeddings AS pe 
JOIN 
  pdfs AS p ON p.id = pe.id
CROSS JOIN 
  tbl 
WHERE city = $city_name
ORDER BY 
  cosineDistance ASC
  LIMIT 5
WITH description = 'Get information about a specific city';

We create another tool, `get_semantics`, which uses the built-in table function `semantics()`, which returns the schemas of all the tables in the database.

In [ ]:
CREATE ENDPOINT get_semantics() AS 
SELECT * FROM semantics()
WITH description = 'Schemas of all the tables';

## Prompt Creation
We create a prompt for our use case based on the ReAct framework.

In [ ]:
CREATE PROMPT cities_prompt (
  system "You are a master data agent. Your task is to provide useful information to the user about a city based on the question.

  Only use information retrieved using the supplied tools.
  
  For doing this, first, fetch the schema of the tables in the database using the get_semantics tool. Understand the schema of the cities table thoroughly to understand the kind of information you can look up using a SQL query on the provided database. If required, generate a valid Clickhouse SQL query to get any required information from the cities table and use the langdb_raw_query to execute that query and get the result.

  You can retrieve information regarding cities using either cities_info_specific and/or cities_info_generic tool. Both the tools use similarity search based on cosine distance to return the most relevant information snippets related to the query.
  1. When you can cities_info_specific tool: If the required city is known or was looked up using a SQL query, use this tool to retrieve information about that specific city.

  2. When you can cities_info_generic tool:
  Use this tool to look up the information when the city is not known and you are not able to use a SQL query to find the required city.
  
  Use the following format:
  Question: the input question you must answer
  Thought: you should always think about what to do
  Action: the action to take, should be be one of the tools
  Action Input: the input to the action
  Observation: the result of the action
  ... (this Thought/Action/Action Input/Observation can repeat N times)
  Thought: I think I have enough information to answer the question. Based on the data retrieved, I can now answer the question.
  Final Answer: the final answer to the original input question with data to support it
  
  Begin!
  Question: {{input}}
  Thought: I need to understand the semantics of the data structure available in the database."
);

## Model Creation
Now, we can create the models that can leverage the tools that were created earlier.

In [ ]:
CREATE MODEL IF NOT EXISTS cities_info_model( 
    provider 'OpenAI',
    model_name 'gpt-3.5-turbo',
    prompt_name 'cities_prompt',
    execution_options (retries 2),
    args ["input"],
    tools ["get_semantics", "cities_info_specific", "cities_info_generic", "langdb_raw_query"]
);

Along with the tools we had created, we have also attached `langdb_raw_query`, a built-in static tool, which allows the model to execute raw SELECT (only) queries on the database.

## Model Execution
Using the created model, we can execute queries which would require the LLM to use both structured (cities table) and unstructured (Wikipedia articles) data through the provided tools.


In [26]:
select cities_info_model('Tell me about the arts and culture of the city with the highest population');

,cities_info_model('Tell me about the arts and culture of the city with the highest population')
0,"The city with the highest population is Tokyo. Tokyo has a vibrant arts and culture scene which includes various museums like the Tokyo National Museum specializing in traditional Japanese art. There are also international and ethnic schools in Tokyo that provide diverse educational programs. The architecture of Tokyo has been influenced by its history, with modern and contemporary styles dominating the urban landscape.\n\nHere are some specific details about Tokyo's arts and culture:\n- Tokyo has many museums, with the Tokyo National Museum being the country's largest museum specializing in traditional Japanese art.\n- There are international and ethnic schools in Tokyo that offer educational programs based on national or international curricula.\n- The architecture in Tokyo has been shaped by its history, with modern and contemporary styles being prevalent due to past earthquakes and firebombing incidents.\n\nOverall, Tokyo offers a rich cultural experience with a mix of traditional and modern elements in its arts, education, and architecture."


In the above query, the model generates a SQL query to find the city with the most populous city and invokes the `langdb_raw_query` tool to execute the generated query. It uses the result from the query, i.e. Tokyo, and invokes the `cities_info_specific` tool to get more information about the arts and culture of the city.

In [28]:
select cities_info_model('Which city conducted the 1964 Summer Olympics');

,cities_info_model('Which city conducted the 1964 Summer Olympics')
0,The city that conducted the 1964 Summer Olympics is Tokyo. The International Olympic Committee (IOC) selected Tokyo as the host of the 2020 Summer Olympic Games as well.


Unlike the previous query, the model invokes the `cities_info_generic` tool to find the required information directly.